In [ ]:
#!/usr/bin/env python3
"""
Translate only the Resume and Job Description columns of a CSV file to Romanian.

Supported source languages:
    en -> Romanian, using Helsinki-NLP/opus-mt-en-ro
    es -> Romanian, using Helsinki-NLP/opus-mt-es-ro

The script preserves:
- every other CSV column unchanged;
- blank lines and the number/order of lines inside each translated cell;
- separators such as ---;
- bullet prefixes such as "- " and "• ";
- the contact-information block at the beginning of each resume;
- URLs, email addresses, and standalone telephone-number lines.

The output is written with UTF-8 BOM (utf-8-sig), which preserves Romanian
characters such as ă, â, î, ș, and ț and is also convenient when opening the
CSV file in Microsoft Excel.

Translations are cached in SQLite. If execution is interrupted, running the
script again continues by reusing translations that were already saved.

All execution settings are defined in the CONFIGURATION section below. The
script mounts Google Drive, reads the CSV from Drive, performs translation on
a Google Colab GPU, saves the result back to Drive, and optionally starts a
browser download of the generated CSV.
"""

from __future__ import annotations

import csv
import os
import re
import shutil
import sqlite3
import sys
import unicodedata
from dataclasses import dataclass
from typing import Iterable, Iterator, Sequence

try:
    from google.colab import drive, files
except ImportError as exc:
    raise RuntimeError(
        "This script is intended to run inside Google Colab."
    ) from exc

import torch
from tqdm.auto import tqdm
from transformers import MarianMTModel, MarianTokenizer


MODEL_BY_SOURCE_LANGUAGE = {
    "en": "Helsinki-NLP/opus-mt-en-ro",
    "es": "Helsinki-NLP/opus-mt-es-ro",
}

DEFAULT_COLUMNS = ("Resume", "Job Description")

# =============================================================================
# CONFIGURATION
# =============================================================================

# Google Drive is mounted at this location in Colab.
DRIVE_MOUNT_POINT = "/content/drive"

# Change only this path to select the CSV database from Google Drive.
# "MyDrive" corresponds to "My Drive" in the Google Drive interface.
INPUT_CSV = "/content/drive/MyDrive/TalentCLEF/translation/job_applicant_dataset(1).csv"

# Folder in Google Drive where the translated CSV and translation cache are saved.
OUTPUT_FOLDER = "/content/drive/MyDrive/TalentCLEF/translation/romanian_output"

# Leave as None to generate: <input_file_name>_ro.csv
OUTPUT_FILE_NAME: str | None = None

# Source language:
#     "en" = English
#     "es" = Spanish
SOURCE_LANGUAGE = "en"

# Names of the only two columns that are translated.
RESUME_COLUMN = "Resume"
JOB_DESCRIPTION_COLUMN = "Job Description"

# Translation settings.
BATCH_SIZE = 16
MAX_SOURCE_TOKENS = 450
MAX_OUTPUT_TOKENS = 512
NUM_BEAMS = 4

# The script requires a Colab GPU.
DEVICE = "cuda"

# Use half precision on the GPU to reduce memory use and improve speed.
USE_FP16 = True

# Set an integer such as 20 to translate only the first rows for testing.
# Use None to translate the complete database.
ROW_LIMIT: int | None = None

# Automatically start a browser download after the file is saved to Drive.
DOWNLOAD_RESULT = True

# Also copy the final CSV to /content before starting the browser download.
LOCAL_DOWNLOAD_FOLDER = "/content/romanian_translation_download"
# =============================================================================

EMAIL_RE = re.compile(r"^[^\s@]+@[^\s@]+\.[^\s@]+$")
URL_RE = re.compile(r"^(?:https?://|www\.)\S+$", re.IGNORECASE)
PHONE_RE = re.compile(r"^\+?[\d\s()./-]{7,}$")
ONLY_STRUCTURE_RE = re.compile(r"^[\W_]+$", re.UNICODE)
BULLET_RE = re.compile(r"^((?:[-•*]|\d+[.)])\s+)(.*)$", re.DOTALL)
SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")

ROMANIAN_DIACRITIC_TRANSLATION = str.maketrans(
    {
        "ş": "ș",
        "Ş": "Ș",
        "ţ": "ț",
        "Ţ": "Ț",
    }
)


@dataclass(frozen=True)
class CellUnit:
    """One preserved or translatable unit from a cell."""

    kind: str  # "raw" or "translate"
    raw: str = ""
    prefix: str = ""
    text: str = ""
    trailing_space: str = ""
    line_ending: str = ""






In [ ]:
def split_line_ending(raw_line: str) -> tuple[str, str]:
    if raw_line.endswith("\r\n"):
        return raw_line[:-2], "\r\n"
    if raw_line.endswith("\n") or raw_line.endswith("\r"):
        return raw_line[:-1], raw_line[-1]
    return raw_line, ""


def normalize_romanian_text(text: str) -> str:
    """
    Normalize Romanian Unicode characters without applying any Latin-1
    encode/decode conversion.
    """
    text = unicodedata.normalize("NFC", text)
    text = text.translate(ROMANIAN_DIACRITIC_TRANSLATION)
    # A translated unit represents one original line, so keep it on one line.
    text = re.sub(r"\s*\r?\n\s*", " ", text)
    return text.strip()


def should_preserve_line(text: str) -> bool:
    stripped = text.strip()

    if not stripped:
        return True
    if stripped == "---":
        return True
    if EMAIL_RE.fullmatch(stripped):
        return True
    if URL_RE.fullmatch(stripped):
        return True
    if PHONE_RE.fullmatch(stripped):
        return True
    if ONLY_STRUCTURE_RE.fullmatch(stripped):
        return True

    return False


def iter_cell_units(text: str, column_name: str, resume_column: str) -> Iterator[CellUnit]:
    """
    Split a cell into line-based units.

    For resumes, all lines before and including the first --- separator are
    kept unchanged. This protects the candidate name and contact information.
    """
    in_resume_contact_block = column_name == resume_column

    for raw_line in text.splitlines(keepends=True):
        body, line_ending = split_line_ending(raw_line)

        if in_resume_contact_block:
            yield CellUnit(kind="raw", raw=raw_line)
            if body.strip() == "---":
                in_resume_contact_block = False
            continue

        if should_preserve_line(body):
            yield CellUnit(kind="raw", raw=raw_line)
            continue

        leading_match = re.match(r"^\s*", body)
        leading_space = leading_match.group(0) if leading_match else ""
        remainder = body[len(leading_space) :]

        bullet_match = BULLET_RE.match(remainder)
        bullet_prefix = ""
        if bullet_match:
            bullet_prefix = bullet_match.group(1)
            remainder = bullet_match.group(2)

        trailing_match = re.search(r"\s*$", remainder)
        trailing_space = trailing_match.group(0) if trailing_match else ""
        core_text = remainder[: len(remainder) - len(trailing_space)] if trailing_space else remainder

        if not core_text.strip():
            yield CellUnit(kind="raw", raw=raw_line)
            continue

        yield CellUnit(
            kind="translate",
            prefix=leading_space + bullet_prefix,
            text=core_text,
            trailing_space=trailing_space,
            line_ending=line_ending,
        )

    # splitlines() returns no units for an empty string.
    if text == "":
        return


def extract_translatable_segments(
    text: str,
    column_name: str,
    resume_column: str,
) -> Iterator[str]:
    for unit in iter_cell_units(text, column_name, resume_column):
        if unit.kind == "translate":
            yield unit.text


def rebuild_cell(
    text: str,
    column_name: str,
    resume_column: str,
    translations: dict[str, str],
) -> str:
    output_parts: list[str] = []

    for unit in iter_cell_units(text, column_name, resume_column):
        if unit.kind == "raw":
            output_parts.append(unit.raw)
            continue

        translated = translations.get(unit.text)
        if translated is None:
            raise KeyError(f"Missing cached translation for: {unit.text[:100]!r}")

        output_parts.append(
            unit.prefix + translated + unit.trailing_space + unit.line_ending
        )

    result = "".join(output_parts)

    # The line structure must remain unchanged.
    if result.count("\n") != text.count("\n"):
        raise RuntimeError(
            "The translated cell no longer has the same number of line breaks."
        )

    return result




In [ ]:
def token_count(tokenizer: MarianTokenizer, text: str) -> int:
    return len(tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"])


def split_oversized_piece_by_words(
    text: str,
    tokenizer: MarianTokenizer,
    max_tokens: int,
) -> list[str]:
    words = text.split()
    if not words:
        return []

    chunks: list[str] = []
    current: list[str] = []

    for word in words:
        candidate = " ".join(current + [word])

        if current and token_count(tokenizer, candidate) > max_tokens:
            chunks.append(" ".join(current))
            current = [word]
        else:
            current.append(word)

        # Extremely long strings without spaces are kept as their own unit.
        if len(current) == 1 and token_count(tokenizer, current[0]) > max_tokens:
            chunks.append(current[0])
            current = []

    if current:
        chunks.append(" ".join(current))

    return chunks


def split_text_for_model(
    text: str,
    tokenizer: MarianTokenizer,
    max_tokens: int,
) -> list[str]:
    """Split one line only when it exceeds the Marian source-token limit."""
    if token_count(tokenizer, text) <= max_tokens:
        return [text]

    sentences = [part.strip() for part in SENTENCE_SPLIT_RE.split(text) if part.strip()]
    if not sentences:
        return split_oversized_piece_by_words(text, tokenizer, max_tokens)

    chunks: list[str] = []
    current: list[str] = []

    for sentence in sentences:
        if token_count(tokenizer, sentence) > max_tokens:
            if current:
                chunks.append(" ".join(current))
                current = []
            chunks.extend(
                split_oversized_piece_by_words(sentence, tokenizer, max_tokens)
            )
            continue

        candidate = " ".join(current + [sentence])
        if current and token_count(tokenizer, candidate) > max_tokens:
            chunks.append(" ".join(current))
            current = [sentence]
        else:
            current.append(sentence)

    if current:
        chunks.append(" ".join(current))

    return chunks




In [ ]:
class TranslationCache:
    def __init__(self, path: str, model_name: str) -> None:
        cache_directory = os.path.dirname(path)
        if cache_directory:
            os.makedirs(cache_directory, exist_ok=True)
        self.connection = sqlite3.connect(path)
        self.model_name = model_name
        self.connection.execute(
            """
            CREATE TABLE IF NOT EXISTS translations (
                model_name TEXT NOT NULL,
                source_text TEXT NOT NULL,
                translated_text TEXT NOT NULL,
                PRIMARY KEY (model_name, source_text)
            )
            """
        )
        self.connection.commit()

    def existing_sources(self) -> set[str]:
        cursor = self.connection.execute(
            "SELECT source_text FROM translations WHERE model_name = ?",
            (self.model_name,),
        )
        return {row[0] for row in cursor}

    def save_many(self, pairs: Iterable[tuple[str, str]]) -> None:
        self.connection.executemany(
            """
            INSERT OR REPLACE INTO translations
                (model_name, source_text, translated_text)
            VALUES (?, ?, ?)
            """,
            (
                (self.model_name, source, translated)
                for source, translated in pairs
            ),
        )
        self.connection.commit()

    def load_all(self) -> dict[str, str]:
        cursor = self.connection.execute(
            """
            SELECT source_text, translated_text
            FROM translations
            WHERE model_name = ?
            """,
            (self.model_name,),
        )
        return dict(cursor.fetchall())

    def close(self) -> None:
        self.connection.close()




In [ ]:
def resolve_device(requested: str) -> torch.device:
    if requested == "cpu":
        return torch.device("cpu")
    if requested == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA was requested, but no CUDA device is available.")
        return torch.device("cuda")
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def translate_model_chunks(
    chunks: Sequence[str],
    tokenizer: MarianTokenizer,
    model: MarianMTModel,
    device: torch.device,
    batch_size: int,
    max_source_tokens: int,
    max_output_tokens: int,
    num_beams: int,
) -> list[str]:
    translated_chunks: list[str] = []

    for start in range(0, len(chunks), batch_size):
        batch = list(chunks[start : start + batch_size])

        encoded = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_source_tokens,
        ).to(device)

        with torch.inference_mode():
            generated = model.generate(
                **encoded,
                max_length=max_output_tokens,
                num_beams=num_beams,
                early_stopping=(num_beams > 1),
            )

        decoded = tokenizer.batch_decode(
            generated,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )
        translated_chunks.extend(normalize_romanian_text(item) for item in decoded)

    return translated_chunks


def translate_source_block(
    source_texts: Sequence[str],
    tokenizer: MarianTokenizer,
    model: MarianMTModel,
    device: torch.device,
    batch_size: int,
    max_source_tokens: int,
    max_output_tokens: int,
    num_beams: int,
) -> list[tuple[str, str]]:
    """
    Translate a manageable block of source lines. Long lines are split into
    smaller chunks and then reconstructed on their original single line.
    """
    all_chunks: list[str] = []
    ownership: list[tuple[int, int]] = []

    for source_index, source in enumerate(source_texts):
        pieces = split_text_for_model(source, tokenizer, max_source_tokens)
        start = len(all_chunks)
        all_chunks.extend(pieces)
        ownership.append((start, len(pieces)))

    translated_chunks = translate_model_chunks(
        all_chunks,
        tokenizer,
        model,
        device,
        batch_size,
        max_source_tokens,
        max_output_tokens,
        num_beams,
    )

    results: list[tuple[str, str]] = []
    for source, (start, count) in zip(source_texts, ownership):
        translated = " ".join(translated_chunks[start : start + count]).strip()
        results.append((source, normalize_romanian_text(translated)))

    return results




In [ ]:
def iter_rows(
    input_path: str,
    limit: int | None,
) -> tuple[list[str], Iterator[dict[str, str]]]:
    file_handle = open(input_path, "r", encoding="utf-8-sig", newline="")
    reader = csv.DictReader(file_handle)

    if reader.fieldnames is None:
        file_handle.close()
        raise ValueError("The CSV file has no header row.")

    def generator() -> Iterator[dict[str, str]]:
        try:
            for index, row in enumerate(reader):
                if limit is not None and index >= limit:
                    break
                yield row
        finally:
            file_handle.close()

    return list(reader.fieldnames), generator()


def collect_unique_segments(
    input_path: str,
    columns: Sequence[str],
    resume_column: str,
    limit: int | None,
) -> tuple[list[str], set[str], int]:
    fieldnames, rows = iter_rows(input_path, limit)

    missing = [column for column in columns if column not in fieldnames]
    if missing:
        raise ValueError(
            f"Missing required column(s): {missing}. "
            f"Available columns: {fieldnames}"
        )

    unique_segments: set[str] = set()
    row_count = 0

    for row in tqdm(rows, desc="Scanning CSV rows"):
        row_count += 1
        for column in columns:
            value = row.get(column, "")
            unique_segments.update(
                extract_translatable_segments(value, column, resume_column)
            )

    return fieldnames, unique_segments, row_count


def write_translated_csv(
    input_path: str,
    output_path: str,
    fieldnames: Sequence[str],
    columns: Sequence[str],
    resume_column: str,
    translations: dict[str, str],
    limit: int | None,
    expected_rows: int,
) -> None:
    output_directory = os.path.dirname(output_path)
    if output_directory:
        os.makedirs(output_directory, exist_ok=True)

    _, rows = iter_rows(input_path, limit)
    written_rows = 0

    with open(output_path, "w", encoding="utf-8-sig", newline="") as output_file:
        writer = csv.DictWriter(
            output_file,
            fieldnames=fieldnames,
            extrasaction="raise",
            quoting=csv.QUOTE_MINIMAL,
        )
        writer.writeheader()

        for row in tqdm(rows, total=expected_rows, desc="Writing Romanian CSV"):
            original_row = dict(row)

            for column in columns:
                row[column] = rebuild_cell(
                    row.get(column, ""),
                    column,
                    resume_column,
                    translations,
                )

            # Verify that no other column was changed.
            for field in fieldnames:
                if field not in columns and row[field] != original_row[field]:
                    raise RuntimeError(
                        f"Unexpected modification in column {field!r}."
                    )

            writer.writerow(row)
            written_rows += 1

    if written_rows != expected_rows:
        raise RuntimeError(
            f"Expected {expected_rows} rows, but wrote {written_rows}."
        )





In [ ]:
def mount_google_drive() -> None:
    """Mount Google Drive when it is not already mounted."""
    drive_root = os.path.join(DRIVE_MOUNT_POINT, "MyDrive")

    if os.path.exists(drive_root):
        print(f"Google Drive is already mounted at: {DRIVE_MOUNT_POINT}")
        return

    print("Mounting Google Drive...")
    drive.mount(DRIVE_MOUNT_POINT)


def prepare_gpu() -> torch.device:
    """Require and report the Google Colab CUDA GPU."""
    if DEVICE != "cuda":
        raise ValueError('For this Colab version, DEVICE must remain "cuda".')

    if not torch.cuda.is_available():
        raise RuntimeError(
            "No GPU is available. In Google Colab, open Runtime > "
            "Change runtime type and select a GPU accelerator, then run again."
        )

    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)

    print(f"GPU available: {gpu_name}")
    print(f"CUDA version: {torch.version.cuda}")

    # TF32 can accelerate supported operations on recent NVIDIA GPUs.
    torch.backends.cuda.matmul.allow_tf32 = True

    return device


def download_result_from_colab(output_path: str) -> None:
    """
    Copy the Drive output to Colab local storage and start a browser download.
    The permanent copy remains stored in Google Drive.
    """
    os.makedirs(LOCAL_DOWNLOAD_FOLDER, exist_ok=True)
    local_path = os.path.join(
        LOCAL_DOWNLOAD_FOLDER,
        os.path.basename(output_path)
    )

    shutil.copy2(output_path, local_path)

    print(f"Local downloadable copy: {local_path}")
    files.download(local_path)




In [ ]:
def main() -> None:
    csv.field_size_limit(sys.maxsize)

    mount_google_drive()

    if SOURCE_LANGUAGE not in MODEL_BY_SOURCE_LANGUAGE:
        raise ValueError(
            f"SOURCE_LANGUAGE must be one of "
            f"{sorted(MODEL_BY_SOURCE_LANGUAGE)}, not {SOURCE_LANGUAGE!r}."
        )

    if BATCH_SIZE < 1:
        raise ValueError("BATCH_SIZE must be at least 1.")
    if MAX_SOURCE_TOKENS < 1:
        raise ValueError("MAX_SOURCE_TOKENS must be at least 1.")
    if MAX_OUTPUT_TOKENS < 1:
        raise ValueError("MAX_OUTPUT_TOKENS must be at least 1.")
    if NUM_BEAMS < 1:
        raise ValueError("NUM_BEAMS must be at least 1.")
    if ROW_LIMIT is not None and ROW_LIMIT < 1:
        raise ValueError("ROW_LIMIT must be None or a positive integer.")

    input_path = INPUT_CSV
    output_folder = OUTPUT_FOLDER
    os.makedirs(output_folder, exist_ok=True)

    if not os.path.exists(input_path):
        raise FileNotFoundError(
            f"Input file not found: {input_path}\n"
            "Update INPUT_CSV in the CONFIGURATION section."
        )

    output_name = (
        OUTPUT_FILE_NAME
        if OUTPUT_FILE_NAME is not None
        else f"{os.path.splitext(os.path.basename(input_path))[0]}_ro.csv"
    )
    output_path = os.path.join(output_folder, output_name)

    columns = (RESUME_COLUMN, JOB_DESCRIPTION_COLUMN)
    model_name = MODEL_BY_SOURCE_LANGUAGE[SOURCE_LANGUAGE]
    cache_name = f"{os.path.splitext(output_name)[0]}.translations.sqlite3"
    cache_path = os.path.join(output_folder, cache_name)

    print(f"Input CSV: {input_path}")
    print(f"Output CSV: {output_path}")
    print(f"Source language: {SOURCE_LANGUAGE}")
    print(f"Translation model: {model_name}")
    print(f"Translation cache: {cache_path}")

    fieldnames, unique_segments, row_count = collect_unique_segments(
        input_path=input_path,
        columns=columns,
        resume_column=RESUME_COLUMN,
        limit=ROW_LIMIT,
    )

    print(f"Rows selected: {row_count}")
    print(f"Unique translatable lines: {len(unique_segments)}")

    device = prepare_gpu()

    tokenizer = MarianTokenizer.from_pretrained(model_name)

    model_dtype = torch.float16 if USE_FP16 else torch.float32
    model = MarianMTModel.from_pretrained(
        model_name,
        torch_dtype=model_dtype,
    )
    model.to(device)
    model.eval()

    print(f"Model precision: {model_dtype}")

    cache = TranslationCache(cache_path, model_name)

    try:
        existing = cache.existing_sources()
        pending = sorted(
            (segment for segment in unique_segments if segment not in existing),
            key=len,
        )

        print(f"Already cached: {len(unique_segments) - len(pending)}")
        print(f"Still to translate: {len(pending)}")

        # Process source lines in moderate blocks so that the script does not
        # keep all tokenized chunks in GPU memory simultaneously.
        source_block_size = max(BATCH_SIZE * 8, 64)

        progress = tqdm(total=len(pending), desc="Translating unique lines")
        for start in range(0, len(pending), source_block_size):
            source_block = pending[start : start + source_block_size]

            translated_pairs = translate_source_block(
                source_texts=source_block,
                tokenizer=tokenizer,
                model=model,
                device=device,
                batch_size=BATCH_SIZE,
                max_source_tokens=MAX_SOURCE_TOKENS,
                max_output_tokens=MAX_OUTPUT_TOKENS,
                num_beams=NUM_BEAMS,
            )

            cache.save_many(translated_pairs)
            progress.update(len(source_block))

        progress.close()

        translations = cache.load_all()

        missing_after_translation = unique_segments - translations.keys()
        if missing_after_translation:
            raise RuntimeError(
                f"{len(missing_after_translation)} translations are missing."
            )

        write_translated_csv(
            input_path=input_path,
            output_path=output_path,
            fieldnames=fieldnames,
            columns=columns,
            resume_column=RESUME_COLUMN,
            translations=translations,
            limit=ROW_LIMIT,
            expected_rows=row_count,
        )
    finally:
        cache.close()

    print("\nTranslation completed successfully.")
    print(f"Permanent Google Drive copy: {output_path}")
    print(
        "Encoding: UTF-8 with BOM. Romanian diacritics are preserved "
        "(ă, â, î, ș, ț)."
    )

    if DOWNLOAD_RESULT:
        download_result_from_colab(output_path)


if __name__ == "__main__":
    main()
